# S2 Pair Panel (1D)

Builds long-format **pair panels** for the S2 universes. The role of this notebook is a
**candidate cache**: it screens first, then materialises panels only for pairs a book could
actually hold.

## Two-pass build

| Pass | Input | Work | Scope |
|------|-------|------|-------|
| 1 — screen | close series only (**no panel**) | EG at `RESEARCH_IS_END` (freeze) and, in rotate scope, at every quarter-end on trailing `L = 252` | all candidates |
| 2 — panel | OHLC frames | `build_pair_panel` incl. **per-bar** `adf_pvalue` | **union of pairs ever selected** |

Rolling `adf_pvalue` exists for exactly one purpose — the `BREAK_STAR` health gate on a pair that is
**actively traded**. Discovery is Engle-Granger, not rolling ADF. So computing per-bar ADF for all
~139 candidates would be wasted work: a pair never selected at any point-in-time rebalance is never
traded, and omitting its panel changes no signal. Both H-004 arms still read one identical panel,
which is what keeps that bake-off single-variable.

## `BOOK_SCOPE`

| Value | Used by | Union covers |
|-------|---------|--------------|
| `freeze_only` | H-001 (scores A–F) | freeze book per universe (≤ 6 pairs each) |
| `freeze_plus_rotate` | H-004 (on `UNIVERSE_STAR` only) | freeze book ∪ every quarterly selection |

Run `freeze_only` before H-001, then re-run `freeze_plus_rotate` for the chosen universe before
H-004. This avoids computing rotate schedules for universes that lose H-001.

**Runtime:** pass 1 is ~139 candidates × up to ~15 screen dates of Engle-Granger; pass 2 is per-bar
ADF on the union only (tens of pairs, not 139).

## Conventions

**Timing contract (S2 — not Default close-fill, not S1 trade-date):** features / signal / decision at
close `t`; orders queued just before open `t+1`; fill at open of `t+1` on both legs. Hedge, z, ADF
and half-life use **closes only**; open / high / low are for fill, stop and PnL path.

**Candidates:** pairs form **only within a leaf pool** via `iter_pool_pairs` over `S2_POOLS`, at any
nesting depth. The manual `KEEP_PAIR_IDS` keep-list is **gone** — selection is deterministic p-value
ranking under caps (global 6, per-pool 2), so there is no discretionary channel here.

**Pair identity:** `pair_id = ticker_y|ticker_x`, oriented by the EG-chosen direction (lower p-value
of `y~x` vs `x~y`).

**No global panel `START_DATE`:** each pair starts on the first date both legs have a valid close
(unbalanced panel is intentional).

**Research IS:** fixed calendar `T` per universe — A `2020-12-31`, B `2022-12-31`, C `2021-12-31`,
D / E / F `2021-12-31`. Registered in `data.processing.s2_universe_pools`. Sealed OOS is `date > T`.

**Panel z:** fixed `Z_WINDOW=60` (adaptive window is H-014, not used here). Rolling `half_life` uses
`HL_WINDOW` and is a diagnostic here — no half-life filter exists before H-008.

## Outputs

| File | Contents |
|------|----------|
| `s2_candidates_{letter}_1d.csv` | screen metrics for **every** candidate at IS end (non-selected stay auditable) |
| `s2_pairs_{letter}_1d.csv` | ranked frozen book under caps |
| `s2_rebalance_{letter}_1d.csv` | quarterly selections (rotate scope only) |
| `s2_panel_{letter}_1d_train.parquet` | union pairs, research IS (`date <= T`) |
| `s2_panel_{letter}_1d_full.parquet` | union pairs, full census |

Hypotheses: `02_research/s2_coint/s2_hypothesis_log.md`. Notes:
`05_strategies/s2_coint/s2_algorithm_notes.md`.

## 0. Imports & Config

In [ ]:
import os
import sys

import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s2_coint.rotation import build_active_schedule, freeze_book
from data.ingestion.equity_fetcher import fetch_ohlcv
from data.processing.s2_coint_store import build_pair_panel
from data.processing.s2_universe import (
    RESEARCH_IS_END_BY_UNIVERSE,
    iter_pool_pairs,
    load_s2_pools,
    pool_of_pair,
    pool_tickers,
    ticker_venue_key,
)
from strategies.s2_coint.book import CAP_GLOBAL, CAP_PER_POOL, DISCOVERY_LOOKBACK_BARS

NOTEBOOK_DIR = os.path.join(ROOT, "01_data", "data_files", "s2_coint")

# "freeze_only" for H-001 (all universes); "freeze_plus_rotate" for H-004 (UNIVERSE_STAR only).
BOOK_SCOPE = "freeze_only"
UNIVERSE_LABELS = ("A", "B", "C", "D", "E", "F")

BAR = "1d"
# Download floor only (not a pair-panel start). Pairs still begin at first mutual close.
FETCH_START = "1990-01-01"
END_DATE = None

RESEARCH_IS_END = dict(RESEARCH_IS_END_BY_UNIVERSE)
COINT_PVALUE_THRESHOLD = 0.05
OLS_WINDOW = 252
Z_WINDOW = 60
HL_WINDOW = 252
INCLUDE_ADF_PVALUE = True
INCLUDE_VARIANCE_JUMP = True

if BOOK_SCOPE not in {"freeze_only", "freeze_plus_rotate"}:
    raise ValueError("BOOK_SCOPE must be 'freeze_only' or 'freeze_plus_rotate'")


def candidates_csv_path(label: str) -> str:
    return os.path.join(NOTEBOOK_DIR, f"s2_candidates_{label}_{BAR}.csv")


def pairs_csv_path(label: str) -> str:
    return os.path.join(NOTEBOOK_DIR, f"s2_pairs_{label}_{BAR}.csv")


def rebalance_csv_path(label: str) -> str:
    return os.path.join(NOTEBOOK_DIR, f"s2_rebalance_{label}_{BAR}.csv")


def panel_train_path(label: str) -> str:
    return os.path.join(NOTEBOOK_DIR, f"s2_panel_{label}_{BAR}_train.parquet")


def panel_full_path(label: str) -> str:
    return os.path.join(NOTEBOOK_DIR, f"s2_panel_{label}_{BAR}_full.parquet")


POOLS = {label: load_s2_pools(label) for label in UNIVERSE_LABELS}
CANDIDATES = {label: iter_pool_pairs(POOLS[label]) for label in UNIVERSE_LABELS}

print("BOOK_SCOPE", BOOK_SCOPE)
for label in UNIVERSE_LABELS:
    print(
        f"{label}: {len(pool_tickers(POOLS[label])):3d} tickers  "
        f"{len(CANDIDATES[label]):3d} candidates  IS end {RESEARCH_IS_END[label]}"
    )
print("total candidates", sum(len(v) for v in CANDIDATES.values()))
print("caps: global", CAP_GLOBAL, "per pool", CAP_PER_POOL, "| L", DISCOVERY_LOOKBACK_BARS)

## 1. Load pools & fetch OHLCV

`isAsian=True` preserves exchange suffixes (`.HK`, `.T`, `.MC`, `.MI`, `.AS`, `.DE`, `.PA`). US
names stay False so share classes such as `BF.B` are hyphenated to Yahoo's `BF-B`. The store never
fetches — it only transforms. Full OHLC frames feed `build_pair_panel`; close views feed the screen.

In [ ]:
ohlc_by_universe: dict[str, dict[str, pd.DataFrame]] = {}
closes_by_universe: dict[str, dict[str, pd.Series]] = {}

for label in UNIVERSE_LABELS:
    ohlc: dict[str, pd.DataFrame] = {}
    closes: dict[str, pd.Series] = {}
    print(f"\n=== Universe {label} ===")
    for ticker in pool_tickers(POOLS[label]):
        is_suffixed = "." in ticker and ticker_venue_key(ticker) != "US"
        panel = fetch_ohlcv(
            ticker,
            start_date=FETCH_START,
            end_date=END_DATE,
            isAsian=is_suffixed,
            interval=BAR,
        )
        if panel is None or panel.empty:
            print(f"WARNING: no OHLCV for {ticker} ({label})")
            continue
        frame = (
            panel.sort_values("date")
            .drop_duplicates("date", keep="last")
            .set_index("date")[["open", "high", "low", "close"]]
            .astype(float)
        )
        frame.index = pd.to_datetime(frame.index)
        frame = frame.dropna(subset=["close"])
        if frame.empty:
            print(f"WARNING: no valid closes for {ticker} ({label})")
            continue
        ohlc[ticker] = frame
        closes[ticker] = frame["close"].copy()
        print(f"{ticker:12} bars={len(frame):5d} venue={ticker_venue_key(ticker)}")
    ohlc_by_universe[label] = ohlc
    closes_by_universe[label] = closes

## 2. Research IS cutoff

Pre-registered calendar `RESEARCH_IS_END` per universe (no fraction of unique dates). Screen and
train both use `date <= T`.

In [ ]:
is_end_by_universe: dict[str, pd.Timestamp] = {}
for label in UNIVERSE_LABELS:
    is_end_by_universe[label] = pd.Timestamp(RESEARCH_IS_END[label])
    print(f"{label}: RESEARCH_IS_END={is_end_by_universe[label].date()}")

## 3. Pass 1 — screen the freeze book (closes only)

`freeze_book` screens the **full IS** (`date <= T`) inside leaf pools, then ranks by p-value under the
caps. No panel is built in this pass.

In [ ]:
freeze_by_universe: dict[str, pd.DataFrame] = {}

for label in UNIVERSE_LABELS:
    closes = closes_by_universe[label]
    if not closes:
        freeze_by_universe[label] = pd.DataFrame()
        print(f"{label}: no closes - skipping")
        continue
    chosen = freeze_book(
        closes,
        POOLS[label],
        is_end=is_end_by_universe[label],
        ols_window=OLS_WINDOW,
        per_pool_cap=CAP_PER_POOL,
        global_cap=CAP_GLOBAL,
        pvalue_threshold=COINT_PVALUE_THRESHOLD,
    )
    freeze_by_universe[label] = chosen
    print(f"{label}: freeze book {len(chosen)} pairs -> {list(chosen.get('pair_id', []))}")
    if not chosen.empty:
        chosen.to_csv(pairs_csv_path(label), index=False)
        print("   saved", pairs_csv_path(label))

### 3.1 Full candidate screen (audit trail)

Every candidate at IS end, selected or not, so a rejected pair can still be inspected later.

In [ ]:
from data.processing.s2_coint_store import screen_pair_cointegration

for label in UNIVERSE_LABELS:
    closes = closes_by_universe[label]
    cands = [(a, b) for a, b in CANDIDATES[label] if a in closes and b in closes]
    if not cands:
        continue
    screened = screen_pair_cointegration(
        closes,
        cands,
        is_end=is_end_by_universe[label],
        ols_window=OLS_WINDOW,
        pvalue_threshold=COINT_PVALUE_THRESHOLD,
    )
    lookup = pool_of_pair(POOLS[label])
    screened["pool"] = [lookup.get(str(p), "?") for p in screened["pair_id"]]
    screened.to_csv(candidates_csv_path(label), index=False)
    n_pass = int((screened["eligible"] & (screened["pvalue"] < COINT_PVALUE_THRESHOLD)).sum())
    print(f"{label}: {len(screened)} candidates, {n_pass} EG passers -> {candidates_csv_path(label)}")

### 3.2 Rotating schedule (only when `BOOK_SCOPE = "freeze_plus_rotate"`)

Quarterly screens on the trailing `L = 252` bars. Each selection is effective at the **next**
session's open, so nothing is tradable on the rebalance bar itself.

In [ ]:
schedule_by_universe: dict[str, pd.DataFrame] = {}

if BOOK_SCOPE == "freeze_plus_rotate":
    for label in UNIVERSE_LABELS:
        closes = closes_by_universe[label]
        if not closes:
            schedule_by_universe[label] = pd.DataFrame()
            continue
        all_dates = pd.DatetimeIndex(sorted({d for s in closes.values() for d in s.index}))
        schedule = build_active_schedule(
            closes,
            POOLS[label],
            all_dates,
            lookback_bars=DISCOVERY_LOOKBACK_BARS,
            ols_window=OLS_WINDOW,
            per_pool_cap=CAP_PER_POOL,
            global_cap=CAP_GLOBAL,
            pvalue_threshold=COINT_PVALUE_THRESHOLD,
        )
        schedule_by_universe[label] = schedule
        if not schedule.empty:
            schedule.to_csv(rebalance_csv_path(label), index=False)
        print(
            f"{label}: {schedule['rebalance_date'].nunique() if not schedule.empty else 0} rebalances, "
            f"{schedule['pair_id'].nunique() if not schedule.empty else 0} distinct pairs"
        )
else:
    print("BOOK_SCOPE='freeze_only' - skipping rotating schedules (run again for H-004).")

## 4. Union of pairs ever selected

This is the **only** set that gets a panel with per-bar ADF. It is PIT-safe: the union decides which
panels are materialised, never which trades fire. Every selection used `date <= T` at its own
rebalance, and a pair never selected is never traded.

In [ ]:
union_by_universe: dict[str, list[tuple[str, str]]] = {}

for label in UNIVERSE_LABELS:
    ids: set[str] = set()
    frozen = freeze_by_universe.get(label, pd.DataFrame())
    if not frozen.empty:
        ids |= {str(p) for p in frozen["pair_id"]}
    sched = schedule_by_universe.get(label, pd.DataFrame())
    if not sched.empty:
        ids |= {str(p) for p in sched["pair_id"]}

    # Orientation follows the EG direction recorded in the screen.
    pairs = [tuple(pid.split("|")) for pid in sorted(ids)]
    union_by_universe[label] = [(y, x) for y, x in pairs]
    print(
        f"{label}: union {len(pairs)} pairs of {len(CANDIDATES[label])} candidates "
        f"({100.0 * len(pairs) / max(len(CANDIDATES[label]), 1):.0f}% get a panel)"
    )

## 5. Pass 2 — build union panels (per-bar ADF)

Per-bar rolling ADF, no stride. This is the expensive step, which is exactly why it runs on the union
rather than on every candidate.

In [ ]:
panels_by_universe: dict[str, pd.DataFrame] = {}

for label in UNIVERSE_LABELS:
    pairs = union_by_universe[label]
    if not pairs:
        print(f"{label}: no selected pairs - skipping panel build")
        panels_by_universe[label] = pd.DataFrame()
        continue
    panel = build_pair_panel(
        ohlc_by_universe[label],
        pairs,
        ols_window=OLS_WINDOW,
        z_window=Z_WINDOW,
        hl_window=HL_WINDOW,
        include_adf_pvalue=INCLUDE_ADF_PVALUE,
        include_variance_jump=INCLUDE_VARIANCE_JUMP,
    )
    panels_by_universe[label] = panel
    if panel.empty:
        print(f"{label}: panel empty")
        continue
    print(
        f"{label}: panel shape={panel.shape}  pairs={panel['pair_id'].nunique()}  "
        f"[{panel['date'].min().date()} .. {panel['date'].max().date()}]"
    )

## 6. Export train / full parquets

Train panel is union-pair rows with `date <= RESEARCH_IS_END`; full is the census.

In [ ]:
for label in UNIVERSE_LABELS:
    panel = panels_by_universe.get(label, pd.DataFrame())
    is_end = is_end_by_universe[label]

    if panel.empty:
        print(f"{label}: empty panel - skip parquet export")
        continue

    dates = pd.to_datetime(panel["date"])
    full_panel = panel.copy()
    train_panel = panel.loc[dates <= is_end].copy()

    full_path = panel_full_path(label)
    train_path = panel_train_path(label)
    full_panel.to_parquet(full_path, index=False)
    train_panel.to_parquet(train_path, index=False)

    n_full_dates = dates.nunique()
    n_train_dates = (
        pd.to_datetime(train_panel["date"]).nunique() if not train_panel.empty else 0
    )
    print(f"saved full census: {full_path}  shape={full_panel.shape}")
    print(f"saved train / research IS: {train_path}  shape={train_panel.shape}")
    print(
        f"  RESEARCH_IS_END={is_end.date()}  "
        f"train_dates={n_train_dates}  full_dates={n_full_dates}"
    )